# Practical 1 — Direct methods in constrained optimization

## T1. Derive the predictor system

For the objective $\tfrac12 x^T Gx+g^Tx$ with symmetric $G$, the constraints are $A^Tx=b$ and $C^Tx\ge d$. Introduce the slack $s=C^Tx-d$ and use multipliers $\gamma$ and $\lambda$ for the equality and inequality constraints, respectively. In the order $z=(x,\gamma,\lambda,s)$, stationarity, equality feasibility, slack feasibility and complementarity give the residual vector

$$
F(z)=\begin{pmatrix}r_L\\r_A\\r_C\\r_s\end{pmatrix}
=\begin{pmatrix}
Gx+g-A\gamma-C\lambda\\
b-A^Tx\\
s+d-C^Tx\\
s\odot\lambda
\end{pmatrix},
$$

where $\odot$ denotes componentwise multiplication. At a solution $F(z)=0$, with $s,\lambda\ge0$ componentwise.

The predictor sets the first-order approximation $F(z)+DF(z)\,\delta z$ equal to zero. The first three residuals are affine. For the fourth, the product rule gives $\delta(s_i\lambda_i)=s_i\,\delta\lambda_i+\lambda_i\,\delta s_i$ to first order. With $S=\mathrm{diag}(s)$ and $\Lambda=\mathrm{diag}(\lambda)$ at the current iterate, the predictor therefore solves

$$
\underbrace{\begin{pmatrix}
G & -A & -C & 0\\
-A^T & 0 & 0 & 0\\
-C^T & 0 & 0 & I\\
0 & 0 & S & \Lambda
\end{pmatrix}}_{M_{\mathrm{KKT}}=DF(z)}
\begin{pmatrix}\delta x\\\delta\gamma\\\delta\lambda\\\delta s\end{pmatrix}
= -\begin{pmatrix}r_L\\r_A\\r_C\\r_s\end{pmatrix}.
$$

## C1. Compute the feasible step size

For a positive slack or multiplier $v_i$, a decreasing component imposes the bound $\alpha\le -v_i/\delta v_i$ so that $v_i+\alpha\delta v_i\ge0$. The largest feasible step up to one is the minimum of $1$ and the bounds from all decreasing components of $s$ and $\lambda$. The update uses $0.95\alpha$ to stay strictly inside the boundary.

In [1]:
import numpy as np


def step_size(lam, dlam, s, ds):
    """Largest step up to one that keeps multipliers and slacks nonnegative."""
    alpha = 1.0
    for values, direction in ((lam, dlam), (s, ds)):
        decreasing = direction < 0
        if np.any(decreasing):
            alpha = min(alpha, np.min(-values[decreasing] / direction[decreasing]))
    return float(alpha)

Take $\lambda=(1,2)$ and $s=(2,1)$. With $\delta\lambda=(0,1)$ and $\delta s=(1,0)$, the full step is feasible. Changing the first slack direction to $-4$ gives $2-4\alpha\ge0$, hence $\alpha=1/2$. If the first multiplier also decreases by $4$, it imposes the tighter bound $\alpha=1/4$.

In [2]:
lam = np.array([1.0, 2.0])
s = np.array([2.0, 1.0])
cases = [
    ("Full step", np.array([0.0, 1.0]), np.array([1.0, 0.0]), 1.0),
    ("Slack limits step", np.array([0.0, 1.0]), np.array([-4.0, 0.0]), 0.5),
    ("Multiplier limits step", np.array([-4.0, 1.0]), np.array([-4.0, 0.0]), 0.25),
]

for label, dlam, ds, expected in cases:
    alpha = step_size(lam, dlam, s, ds)
    assert alpha == expected
    assert np.all(lam + 0.95 * alpha * dlam > 0)
    assert np.all(s + 0.95 * alpha * ds > 0)
    print(f"{label}: alpha = {alpha:g}")

Full step: alpha = 1
Slack limits step: alpha = 0.5
Multiplier limits step: alpha = 0.25


In the slack-limited case the undamped step reaches $s_1=0$. The damped update leaves $s_1=0.1$, as required for the next interior-point iteration.

## C2. Solve the box problem with the full KKT matrix

Take $G=I$, $C=[I,\,-I]$ and $d=-10\,\mathbf{1}$. Then $C^Tx\ge d$ gives $-10\le x_i\le10$. Completing the square gives $f(x)=\tfrac12\|x+g\|_2^2-\tfrac12\|g\|_2^2$, so the exact solution is the componentwise projection of $-g$ onto the box.

Dropping the equality row and column from T1 leaves the unknowns $(\delta x,\delta\lambda,\delta s)$. At each iterate the predictor solves the full system with right-hand side $-(r_L,r_C,r_s)$. The corrector uses the same matrix and adds $\sigma\mu\mathbf{1}-\delta s_{\rm aff}\odot\delta\lambda_{\rm aff}$ to the last block of that right-hand side.

Use the infinity norm for the residuals. Following the brief, stop when **any** of $\|r_L\|_\infty$, $\|r_C\|_\infty$ or $\mu=s^T\lambda/m$ falls below $10^{-16}$, or when the iteration count exceeds 100. Report all three and $\|r_s\|_\infty$ at the returned point.

In [3]:
def box_problem(n, seed):
    G = np.eye(n)
    g = np.random.default_rng(seed).standard_normal(n)
    C = np.hstack((G, -G))
    d = np.full(2 * n, -10.0)
    return G, g, C, d


def full_kkt_matrix(G, C, lam, s):
    n, m = C.shape
    return np.block([
        [G,                 -C,                np.zeros((n, m))],
        [-C.T,              np.zeros((m, m)),  np.eye(m)],
        [np.zeros((m, n)),  np.diag(s),        np.diag(lam)],
    ])


def full_direction(G, C, lam, s):
    K = full_kkt_matrix(G, C, lam, s)
    n, m = C.shape

    def solve(a, b, c):
        direction = np.linalg.solve(K, np.concatenate((a, b, c)))
        return np.split(direction, [n, n + m])

    return solve


def solve_qp(G, g, C, d, prepare_direction, A=None, b=None):
    n, m = C.shape
    x = np.zeros(n)
    lam = np.ones(m)
    s = np.ones(m)
    if A is None:
        variables = (x, lam, s)
        matrices = (G, C)
    else:
        gamma = np.ones(b.size)
        variables = (x, gamma, lam, s)
        matrices = (G, A, C)
    epsilon = 1e-16
    iterations = 0

    while True:
        r_L = G @ x + g - C @ lam
        r_C = s + d - C.T @ x
        r_s = s * lam
        mu = s @ lam / m
        criteria = {}
        if A is None:
            rhs = (-r_L, -r_C)
        else:
            r_L -= A @ gamma
            r_A = b - A.T @ x
            rhs = (-r_L, -r_A, -r_C)
            criteria["||r_A||_inf"] = np.linalg.norm(r_A, np.inf)
        criteria.update({
            "||r_L||_inf": np.linalg.norm(r_L, np.inf),
            "||r_C||_inf": np.linalg.norm(r_C, np.inf),
            "mu": mu,
        })
        stop = [name for name, value in criteria.items() if value < epsilon]
        if iterations > 100:
            stop.append("iteration limit")
        if stop:
            residuals = {**criteria, "||r_s||_inf": np.linalg.norm(r_s, np.inf)}
            return (*variables, iterations, stop, residuals)

        solve_direction = prepare_direction(*matrices, lam, s)
        affine = solve_direction(*rhs, -r_s)
        dlam_aff, ds_aff = affine[-2:]
        alpha_aff = step_size(lam, dlam_aff, s, ds_aff)
        mu_aff = (s + alpha_aff * ds_aff) @ (lam + alpha_aff * dlam_aff) / m
        sigma = (mu_aff / mu) ** 3

        corrected = -r_s - ds_aff * dlam_aff + sigma * mu
        direction = solve_direction(*rhs, corrected)
        dlam, ds = direction[-2:]
        step = 0.95 * step_size(lam, dlam, s, ds)
        for value, delta in zip(variables, direction, strict=True):
            value += step * delta
        iterations += 1


def solve_box(G, g, C, d, prepare_direction=full_direction):
    return solve_qp(G, g, C, d, prepare_direction)


def solve_full_kkt(G, g, C, d):
    return solve_box(G, g, C, d, full_direction)

In [4]:
G, g, C, d = box_problem(n=8, seed=42)
x, lam, s, iterations, stop, residuals = solve_full_kkt(G, g, C, d)
x_exact = np.clip(-g, -10.0, 10.0)
error = np.linalg.norm(x - x_exact, np.inf)

print(f"Iterations: {iterations}. Stopping condition: {', '.join(stop)}")
for name, value in residuals.items():
    print(f"{name} = {value:.3e}")
print(f"Maximum coordinate error = {error:.3e}")

assert error < 1e-12
assert np.all(C.T @ x >= d)
assert np.all(lam > 0) and np.all(s > 0)

Iterations: 14. Stopping condition: ||r_L||_inf
||r_L||_inf = 8.930e-17
||r_C||_inf = 6.661e-16
mu = 1.612e-13
||r_s||_inf = 3.848e-13
Maximum coordinate error = 1.754e-14


For this draw, $-g$ lies inside the box. The maximum coordinate error is below $10^{-12}$ and all box constraints hold. The stationarity residual triggers the stop while the mean complementarity $\mu$ is still above the stopping tolerance, as allowed by the brief.

## C3. Time the direct solver

For each dimension, generate one box problem with seed 42 and retain it in `box_inputs` for the comparison in C4. Time three complete calls to the C2 solver with `perf_counter` and report their median. The timed interval includes KKT assembly, both dense solves per iteration and the stopping checks. Generate inputs, check the exact solution and compute the condition number outside that interval.

In [5]:
from time import perf_counter

import pandas as pd

dimensions = (8, 16, 32, 64)
box_inputs = {n: box_problem(n, seed=42) for n in dimensions}


def benchmark_box(G, g, C, d, prepare_direction):
    n = g.size
    times = []
    for repeat in range(3):
        start = perf_counter()
        result = solve_box(G, g, C, d, prepare_direction)
        times.append(perf_counter() - start)

    x, lam, s, iterations, stop, residuals = result
    error = np.linalg.norm(x - np.clip(-g, -10.0, 10.0), np.inf)
    assert error < 1e-12
    assert np.all(C.T @ x >= d) and np.all(s > 0) and np.all(lam > 0)
    return {
        "n": n,
        "median (ms)": 1000 * np.median(times),
        "iterations": iterations,
        "max error": error,
        **residuals,
        "stop": ", ".join(stop),
    }

baseline_rows = [benchmark_box(*problem, full_direction) for problem in box_inputs.values()]
baseline = pd.DataFrame(baseline_rows).set_index("n")
print(baseline.to_string(float_format=lambda value: f"{value:.3e}"))

    median (ms)  iterations  max error  ||r_L||_inf  ||r_C||_inf        mu  ||r_s||_inf         stop
n                                                                                                   
8     1.306e+00          14  1.754e-14    8.930e-17    6.661e-16 1.612e-13    3.848e-13  ||r_L||_inf
16    2.240e+00          16  0.000e+00    4.384e-17    8.812e-16 3.833e-16    9.644e-16  ||r_L||_inf
32    4.310e+00          14  2.753e-14    9.999e-17    9.437e-16 2.489e-13    4.989e-13  ||r_L||_inf
64    1.746e+01          16  0.000e+00    6.826e-17    8.882e-16 6.171e-16    1.254e-15  ||r_L||_inf


For a conditioning reference, use the full $5n \times 5n$ KKT matrix for $n=8$ at the initial iterate. Here $s=\lambda=\mathbf{1}$. This fixes the iterate for comparing formulations later, since the diagonal blocks change as the solver advances.

In [6]:
G, g, C, d = box_inputs[8]
ones = np.ones(d.size)
K_initial = full_kkt_matrix(G, C, ones, ones)
print(f"Initial full KKT condition number (2-norm, n=8): {np.linalg.cond(K_initial, 2):.3f}")

Initial full KKT condition number (2-norm, n=8): 3.726


The full KKT matrix grows from order 40 to 320. Each iteration therefore requires two larger dense solves even though the iteration counts are similar. The error column checks the returned coordinates independently of the stopping criterion.

## T2. Eliminate variables from the inequality KKT system

Write $(a,b,c)$ for the three blocks of the right-hand side. For the predictor they are $(-r_L,-r_C,-r_s)$, and the corrector changes only $c$. Both directions satisfy
$$
G\delta x-C\delta\lambda=a,\qquad
-C^T\delta x+\delta s=b,\qquad
S\delta\lambda+\Lambda\delta s=c.
$$

**Strategy 1.** Since $\lambda_i>0$, the third equation gives $\delta s=\Lambda^{-1}(c-S\delta\lambda)$. Substitution in the second equation yields
$$
\begin{pmatrix}G&-C\\-C^T&-\Lambda^{-1}S\end{pmatrix}
\begin{pmatrix}\delta x\\\delta\lambda\end{pmatrix}
=\begin{pmatrix}a\\ b-\Lambda^{-1}c\end{pmatrix}.
$$
The diagonal block $-\Lambda^{-1}S$ is negative definite at an interior iterate. With symmetric $G$, the reduced matrix is symmetric. A pivoted $LDL^T$ factorization gives the direction when this matrix is nonsingular. Recover $\delta s$ from the third equation.

**Strategy 2.** From the second equation, $\delta s=b+C^T\delta x$. Set $t=S^{-1}(c-\Lambda b)$ and $W=S^{-1}\Lambda$. Substituting into the third equation gives $\delta\lambda=t-WC^T\delta x$. The first equation then becomes
$$
H\delta x=a+Ct,\qquad H=G+CWC^T.
$$
After the Cholesky solve for $\delta x$, recover the two eliminated directions from the formulas above.

For $s_i,\lambda_i>0$, $W$ is positive diagonal. For symmetric $G\succeq0$, $v^THv=v^TGv+(C^Tv)^TW(C^Tv)$ is positive for all $v\ne0$ precisely when $\ker G\cap\ker C^T=\{0\}$. Under this condition Cholesky applies to $H$. This is also the Schur complement of $-\Lambda^{-1}S$ in Strategy 1, so that system is nonsingular and indefinite. The box problem satisfies the condition because $G=I$.

## C4. Implement and compare the reduced solves

At each iterate, factor the reduced matrix once and use its factors for both right-hand sides in the C2 loop. For pivoted $LDL^T$, SciPy supplies a row permutation and $1\times1$ or $2\times2$ pivots. Apply the permutation to the right-hand side, solve through $L$, $D$ and $L^T$, then restore the original variable order.

In [7]:
from scipy.linalg import cho_factor, cho_solve, ldl, solve_triangular


def factor_ldlt(matrix):
    lu, D, perm = ldl(matrix)
    L = lu[perm, :]

    def solve(rhs):
        y = solve_triangular(L, rhs[perm], lower=True, unit_diagonal=True)
        i = 0
        while i < y.size:
            if i + 1 < y.size and D[i + 1, i] != 0:
                block = slice(i, i + 2)
                y[block] = np.linalg.solve(D[block, block], y[block])
                i += 2
            else:
                y[i] /= D[i, i]
                i += 1
        result = np.empty_like(y)
        result[perm] = solve_triangular(L.T, y, unit_diagonal=True)
        return result

    return solve


def ldlt_matrix(G, C, lam, s):
    return np.block([[G, -C], [-C.T, -np.diag(s / lam)]])


def ldlt_direction(G, C, lam, s):
    solve_reduced = factor_ldlt(ldlt_matrix(G, C, lam, s))

    def solve(a, b, c):
        reduced = solve_reduced(np.concatenate((a, b - c / lam)))
        dx, dlam = np.split(reduced, [G.shape[0]])
        ds = (c - s * dlam) / lam
        return dx, dlam, ds

    return solve


def cholesky_matrix(G, C, lam, s):
    return G + (C * (lam / s)) @ C.T


def cholesky_direction(G, C, lam, s):
    factors = cho_factor(cholesky_matrix(G, C, lam, s), lower=True)

    def solve(a, b, c):
        t = (c - lam * b) / s
        dx = cho_solve(factors, a + C @ t)
        ds = b + C.T @ dx
        dlam = (c - lam * ds) / s
        return dx, dlam, ds

    return solve

### Direction check

Use the eight-variable C3 problem at a common interior iterate with unequal slacks and multipliers, so exchanging $S$ and $\Lambda$ would affect the result. Form both right-hand sides with the full-system predictor. Compare all three blocks of each recovered direction with the direct solution.

In [8]:
G, g, C, d = box_inputs[8]
x = np.zeros(g.size)
s = np.linspace(1, 3, d.size)
lam = np.linspace(2, 1, d.size)
a, b, c = -(G @ x + g - C @ lam), -(s + d - C.T @ x), -s * lam
direct = full_direction(G, C, lam, s)
affine = direct(a, b, c)
_, dlam_aff, ds_aff = affine
alpha = step_size(lam, dlam_aff, s, ds_aff)
mu = s @ lam / s.size
mu_aff = (s + alpha * ds_aff) @ (lam + alpha * dlam_aff) / s.size
corrected = c - ds_aff * dlam_aff + (mu_aff / mu)**3 * mu
reference = direct(a, b, corrected)

reduced_methods = {"LDLT": ldlt_direction, "Cholesky": cholesky_direction}
for name, prepare in reduced_methods.items():
    solve = prepare(G, C, lam, s)
    for stage, rhs_last, expected in (("predictor", c, affine), ("corrector", corrected, reference)):
        difference = np.concatenate(solve(a, b, rhs_last)) - np.concatenate(expected)
        error = np.linalg.norm(difference, np.inf)
        assert error < 1e-12
        print(f"{name} {stage}: maximum direction error = {error:.2e}")

# A small indefinite matrix exercises the row permutation and a 2-by-2 pivot.
pivot_example = np.array([[0., 1., 4.], [1., 2., 0.], [4., 0., 0.]])
rhs = np.array([1., 2., 3.])
np.testing.assert_allclose(pivot_example @ factor_ldlt(pivot_example)(rhs), rhs)

LDLT predictor: maximum direction error = 3.55e-15
LDLT corrector: maximum direction error = 1.07e-14
Cholesky predictor: maximum direction error = 3.55e-15
Cholesky corrector: maximum direction error = 1.23e-14


### Timing, accuracy and conditioning

Use the C3 timing routine on the same inputs and include the full-solve medians measured in C3. For conditioning, evaluate each matrix at the C3 reference iterate, $n=8$, $s=\lambda=\mathbf{1}$.

In [9]:
rows = []
for name, prepare in reduced_methods.items():
    for problem in box_inputs.values():
        rows.append({"method": name, **benchmark_box(*problem, prepare)})

comparison = pd.concat([baseline.reset_index().assign(method="Full"), pd.DataFrame(rows)])
comparison = comparison.set_index(["n", "method"]).sort_index()
print(comparison.to_string(float_format=lambda value: f"{value:.3e}"))

G, g, C, d = box_inputs[8]
ones = np.ones(d.size)
for name, assemble in (("Full", full_kkt_matrix), ("LDLT", ldlt_matrix), ("Cholesky", cholesky_matrix)):
    matrix = assemble(G, C, ones, ones)
    print(f"{name}: order {len(matrix)}, initial 2-norm condition number {np.linalg.cond(matrix):.3f}")

             median (ms)  iterations  max error  ||r_L||_inf  ||r_C||_inf        mu  ||r_s||_inf         stop
n  method                                                                                                    
8  Cholesky    9.761e-01          14  1.754e-14    8.930e-17    6.661e-16 1.612e-13    3.848e-13  ||r_L||_inf
   Full        1.306e+00          14  1.754e-14    8.930e-17    6.661e-16 1.612e-13    3.848e-13  ||r_L||_inf
   LDLT        2.275e+00          14  1.754e-14    8.930e-17    1.943e-15 1.612e-13    3.848e-13  ||r_L||_inf
16 Cholesky    1.096e+00          16  0.000e+00    4.384e-17    8.812e-16 3.833e-16    9.644e-16  ||r_L||_inf
   Full        2.240e+00          16  0.000e+00    4.384e-17    8.812e-16 3.833e-16    9.644e-16  ||r_L||_inf
   LDLT        2.552e+00          16  0.000e+00    4.384e-17    3.109e-15 3.833e-16    9.644e-16  ||r_L||_inf
32 Cholesky    1.164e+00          14  2.753e-14    9.999e-17    9.437e-16 2.489e-13    4.989e-13  ||r_L||_inf
   Full   

The reconstructed directions agree with the full solve to roundoff. All three methods take the same number of iterations for each size and reach the same coordinate accuracy. At $n=64$, Cholesky solves a system of order 64, LDLᵀ one of order 192, and the full method one of order 320. Cholesky has the lowest median time in this experiment.

At the initial iterate $W=I$ and $CC^T=2I$, hence $H=3I$ and $\kappa_2(H)=1$. This explains the measured Cholesky condition number for this iterate. Later diagonal weights depend on the individual slacks and multipliers.

## C5. Solve the supplied problems with equality constraints

Read the one-based coordinate files into arrays of the stated shapes. The Hessian is stored as the upper triangle in `G_2.dad`. Reflect its strict upper triangle to form the symmetric matrix and read the linear term separately from `g.dad`.

Use the four-block KKT matrix derived in T1. Compared with C2, stationarity includes $-A\gamma$, the equality row is $-A^T\delta x=-r_A$, and $\gamma$ starts at $\mathbf{1}$. The predictor and corrector use the same step-size and stopping rules. Each calls `numpy.linalg.solve` on the full dense matrix, of order $n+p+2m$.

In [10]:
from pathlib import Path


def read_coordinates(path, shape):
    entries = np.loadtxt(path, ndmin=2)
    indices = entries[:, :-1].astype(int).T - 1
    values = np.zeros(shape)
    values[tuple(indices)] = entries[:, -1]
    return values


def load_qp(folder, n, p, m):
    folder = Path(folder)
    upper = read_coordinates(folder / "G_2.dad", (n, n))
    G = upper + np.triu(upper, k=1).T
    g = read_coordinates(folder / "g.dad", (n,))
    A = read_coordinates(folder / "A.dad", (n, p))
    b = read_coordinates(folder / "b.dad", (p,))
    C = read_coordinates(folder / "C.dad", (n, m))
    d = read_coordinates(folder / "d.dad", (m,))
    return G, g, A, b, C, d


def general_full_matrix(G, A, C, lam, s):
    n, p = A.shape
    m = s.size
    return np.block([
        [G,                -A,               -C,                 np.zeros((n, m))],
        [-A.T,             np.zeros((p, p)), np.zeros((p, m)),   np.zeros((p, m))],
        [-C.T,             np.zeros((m, p)), np.zeros((m, m)),   np.eye(m)],
        [np.zeros((m, n)), np.zeros((m, p)), np.diag(s),         np.diag(lam)],
    ])


def general_full_direction(G, A, C, lam, s):
    K = general_full_matrix(G, A, C, lam, s)
    n, p = A.shape
    m = s.size

    def solve(a, b, c, e):
        delta = np.linalg.solve(K, np.concatenate((a, b, c, e)))
        return np.split(delta, [n, n + p, n + p + m])

    return solve

### Check the two supplied objectives

Run the smaller instance first. Time each solver call once, then measure the objective, all KKT residuals and the largest violation of $C^Tx\ge d$. The PDF gives nine significant digits for each objective, so compare the results at that precision.

In [11]:
def measure_general(problem, reference, prepare_direction):
    G, g, A, b, C, d = problem
    start = perf_counter()
    x, gamma, lam, s, iterations, stop, residuals = solve_qp(
        G, g, C, d, prepare_direction, A=A, b=b
    )
    seconds = perf_counter() - start
    objective = 0.5 * x @ (G @ x) + g @ x
    violation = max(0.0, np.max(d - C.T @ x))
    assert np.all(lam > 0) and np.all(s > 0)
    return x, {
        "seconds": seconds, "iterations": iterations, "objective": objective,
        "reference": reference, "abs difference": abs(objective - reference),
        "ineq violation": violation, **residuals, "stop": ", ".join(stop),
    }


instances = [
    ("optpr1", "optpr1-20261110", 100, 50, 200, 1.15907181e4),
    ("optpr2", "optpr2-20261110", 1000, 500, 2000, 1.08751157e6),
]
general_inputs = {}
full_results = {}
full_solutions = {}

for name, folder, n, p, m, reference in instances:
    problem = load_qp(folder, n, p, m)
    general_inputs[name] = problem
    full_solutions[name], full_results[name] = measure_general(problem, reference, general_full_direction)

full_table = pd.DataFrame.from_dict(full_results, orient="index")
print(full_table.T.to_string(float_format=lambda value: f"{value:.10g}"))

                        optpr1          optpr2
seconds            0.103073792     63.21045029
iterations                  24              28
objective          11590.71812     1087511.567
reference           11590.7181      1087511.57
abs difference 1.942676499e-05  0.002678499091
ineq violation 2.359223927e-15 1.987299214e-14
||r_A||_inf    1.776356839e-15 4.440892099e-14
||r_L||_inf    3.410605132e-13 7.821654435e-11
||r_C||_inf    2.359223927e-15 1.987299214e-14
mu             2.775176549e-17 6.799266384e-17
||r_s||_inf    6.175534453e-17 1.302260386e-14
stop                        mu              mu


Both objectives round to the nine significant digits given in the PDF. The largest equality residual and inequality violation occur on optpr2, at $4.44\times10^{-14}$ and $1.99\times10^{-14}$ respectively. Both runs stop on mean complementarity $\mu$. On optpr2, the stationarity residual is still $7.82\times10^{-11}$ and the largest individual complementarity product is $1.30\times10^{-14}$ at that point. The larger system takes 28 iterations versus 24 on optpr1, with a tenfold increase in KKT matrix order from 550 to 5,500.

## T3. Derive the symmetric system with equality constraints

Write $(a,b,c,e)$ for the four right-hand-side blocks of T1, with $e$ denoting the complementarity right-hand side for either substep. At an interior iterate $\lambda_i>0$, so the fourth row gives
$$
\delta s=\Lambda^{-1}(e-S\delta\lambda).
$$
Substituting in the third row, $-C^T\delta x+\delta s=c$, yields $-C^T\delta x-\Lambda^{-1}S\delta\lambda=c-\Lambda^{-1}e$. Keeping the stationarity and equality rows gives
$$
\begin{pmatrix}
G&-A&-C\\
-A^T&0&0\\
-C^T&0&-\Lambda^{-1}S
\end{pmatrix}
\begin{pmatrix}\delta x\\\delta\gamma\\\delta\lambda\end{pmatrix}
=
\begin{pmatrix}a\\b\\c-\Lambda^{-1}e\end{pmatrix}.
$$

The off-diagonal blocks are transposes in pairs, $G$ is symmetric and $\Lambda^{-1}S$ is diagonal. The matrix is therefore symmetric. A pivoted $LDL^T$ solve gives the three directions when the system is nonsingular. Recover $\delta s$ from the fourth-row formula above by componentwise division.

## C6. Compare pivoted LDLᵀ with the full KKT solve

Reuse the factorization routine from C4 and the general predictor–corrector loop from C5. Both substeps use the same reduced factors within an iteration.

In [12]:
def general_ldlt_matrix(G, A, C, lam, s):
    p, m = A.shape[1], s.size
    return np.block([
        [G,    -A,               -C],
        [-A.T, np.zeros((p, p)), np.zeros((p, m))],
        [-C.T, np.zeros((m, p)), -np.diag(s / lam)],
    ])


def general_ldlt_direction(G, A, C, lam, s):
    n, p = A.shape
    solve_reduced = factor_ldlt(general_ldlt_matrix(G, A, C, lam, s))

    def solve(a, b, c, e):
        reduced = solve_reduced(np.concatenate((a, b, c - e / lam)))
        dx, dgamma, dlam = np.split(reduced, [n, n + p])
        ds = (e - s * dlam) / lam
        return dx, dgamma, dlam, ds

    return solve

### Direction and conditioning checks on optpr1

At the prescribed initial point $x=0$, $\gamma=\lambda=s=\mathbf{1}$, compare all four blocks of both directions against C5. Use the full predictor to form a common corrector right-hand side. At the same point, compute each system's 2-norm condition number outside the timed solves.

In [13]:
G, g, A, b, C, d = general_inputs["optpr1"]
gamma = np.ones(b.size)
lam = np.ones(d.size)
s = np.ones(d.size)
rhs = (A @ gamma + C @ lam - g, -b, -(s + d))
e = -s * lam
direct = general_full_direction(G, A, C, lam, s)
reduced = general_ldlt_direction(G, A, C, lam, s)
affine = direct(*rhs, e)
dlam_aff, ds_aff = affine[-2:]
alpha = step_size(lam, dlam_aff, s, ds_aff)
mu = s @ lam / s.size
mu_aff = (s + alpha * ds_aff) @ (lam + alpha * dlam_aff) / s.size
corrected = e - ds_aff * dlam_aff + (mu_aff / mu)**3 * mu

for stage, last, expected in (("predictor", e, affine), ("corrector", corrected, direct(*rhs, corrected))):
    expected = np.concatenate(expected)
    actual = np.concatenate(reduced(*rhs, last))
    relative_error = np.linalg.norm(actual - expected, np.inf) / np.linalg.norm(expected, np.inf)
    assert relative_error < 1e-10
    print(f"{stage} relative direction error: {relative_error:.3e}")

for method, assemble in (("Full", general_full_matrix), ("LDLT", general_ldlt_matrix)):
    matrix = assemble(G, A, C, lam, s)
    print(f"{method}: order {len(matrix)}, initial condition number {np.linalg.cond(matrix, 2):.4e}")

predictor relative direction error: 1.203e-14
corrector relative direction error: 2.661e-14
Full: order 550, initial condition number 1.2466e+06
LDLT: order 350, initial condition number 1.2464e+06


### Results on both datasets

Use the same starting point, stopping rule and timing boundaries as C5. Each time is one complete solver run. Compare the two primal solutions coordinate by coordinate and compare each objective with the PDF at its printed precision.

In [14]:
reduced_results = {}
for name, folder, n, p, m, reference in instances:
    x, row = measure_general(general_inputs[name], reference, general_ldlt_direction)
    reduced_results[name] = row
    difference = np.linalg.norm(x - full_solutions[name], np.inf)
    print(f"{name}: maximum coordinate difference from C5 = {difference:.3e}")
    for objective in (row["objective"], full_results[name]["objective"]):
        assert format(objective, ".9g") == format(reference, ".9g")

reduced_table = pd.DataFrame.from_dict(reduced_results, orient="index")
general_comparison = pd.concat({"Full": full_table, "LDLT": reduced_table}, names=["method", "dataset"])
general_comparison = general_comparison.swaplevel().sort_index()
print(general_comparison[["seconds", "iterations", "objective", "ineq violation", "mu", "stop"]]
      .to_string(float_format=lambda value: f"{value:.10g}"))
print(general_comparison.filter(like="||").to_string(float_format=lambda value: f"{value:.3e}"))

optpr1: maximum coordinate difference from C5 = 8.021e-15


optpr2: maximum coordinate difference from C5 = 4.929e-14
                   seconds  iterations   objective  ineq violation              mu stop
dataset method                                                                         
optpr1  Full   0.103073792          24 11590.71812 2.359223927e-15 2.775176549e-17   mu
        LDLT   0.039276125          22 11590.71812 1.443289932e-15  4.44314673e-17   mu
optpr2  Full   63.21045029          28 1087511.567 1.987299214e-14 6.799266384e-17   mu
        LDLT   11.50183475          28 1087511.567  1.91235916e-14 5.003402776e-17   mu
                ||r_A||_inf  ||r_L||_inf  ||r_C||_inf  ||r_s||_inf
dataset method                                                    
optpr1  Full      1.776e-15    3.411e-13    2.359e-15    6.176e-17
        LDLT      1.776e-15    2.274e-13    9.548e-15    9.650e-17
optpr2  Full      4.441e-14    7.822e-11    1.987e-14    1.302e-14
        LDLT      2.309e-14    1.091e-10    4.093e-14    9.741e-15


On optpr2, both methods take 28 iterations and reproduce the reference objective to nine significant digits. Their largest coordinate difference is $4.93\times10^{-14}$. The reduced solve takes 11.5 s versus 63.2 s for the full solve in this run. The largest final stationarity residual is $1.09\times10^{-10}$ for LDLᵀ, even though both methods satisfy the mean complementarity stopping criterion.

On optpr1, the initial 2-norm condition numbers of both matrices are about $1.25\times10^6$. Eliminating the slack variables reduces the matrix order from 550 to 350 on optpr1 and from 5,500 to 3,500 on optpr2. LDLᵀ also reuses the same factors for the corrector. Both changes reduce the linear algebra work per iteration.

Slack recovery affected termination in a trial on optpr1. Using the algebraically equivalent third-row formula $\delta s=c+C^T\delta x$ reached the iteration limit with $\mu$ near $10^{-13}$. Using the fourth-row formula derived in T3 brought $\mu$ below the prescribed tolerance. Comparing the full directions at the initial iterate checks the elimination signs, while the final feasibility and complementarity measures check what happens near the solution.